# 🚀 Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**

El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabaja con múltiples datasets de la empresa:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar la calidad de la información: limpieza y validación de datos en Python.

2. 💰 Analizar la rentabilidad: cálculo de revenue, costos y profit neto.

3. 📉 Entender el comportamiento: diseño de un embudo (funnel) de conversión.

4. 🔄 Medir la lealtad del cliente: análisis de retención por cohortes.

5. 🧪 Validar el impacto de cambios: aplicación de pruebas de hipótesis (Test Z).

6. 📊 Comunicar los hallazgos: consolidación visual mediante un dashboard ejecutivo en Power BI.

El análisis de datos culminará en insights para responder preguntas clave y proponer **recomendaciones accionables**.

---

# 📥 Importación de datasets y validación de la calidad de los datos

**Objetivo:** Cargar las librerías necesarias y extraer los datos originales en DataFrames para realizar una exploración inicial de su estructura, tamaño y tipos de variables.

In [1]:
# Importación de librerías
import pandas as pd
from sqlalchemy import create_engine
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

In [2]:
# Carga archivos
orders = pd.read_csv('/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('/datasets/rappiplus_marketing_spend.csv')

In [3]:
print("--- ORDERS ---")
display(orders.head())
display(orders.info())

--- ORDERS ---


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


None

In [4]:
print("\n--- CATALOG ---")
display(catalog.head())
display(catalog.info())


--- CATALOG ---


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


None

In [5]:
print("\n--- MARKETING ---")
display(marketing.head())
display(marketing.info())


--- MARKETING ---


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


None

---

# 🧹 Revisión y calidad de datos

**Objetivo:** Identificar y procesar valores nulos, registros duplicados e inconsistencias, así como estandarizar los formatos de texto y fecha, para garantizar una base de información íntegra y confiable.

In [6]:
# 1. Eliminar duplicados 
orders = orders.drop_duplicates()
catalog = catalog.drop_duplicates()
marketing = marketing.drop_duplicates()

# 2. Manejo de NULLS
orders = orders.dropna(subset=['cantidad', 'precio_unitario', 'monto_descuento'])
columnas_categoricas_orders = ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']
for col in columnas_categoricas_orders:
    orders[col] = orders[col].fillna('Unknown')

marketing['canal'] = marketing['canal'].fillna('Unknown')

In [7]:
# 3. Validación y conversión de fechas al formato (datetime)
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

# 4. Revisión de variables numéricas (no negativos o ceros inválidos)
orders = orders[(orders['cantidad'] > 0) & (orders['precio_unitario'] > 0) & (orders['monto_descuento'] >= 0)]
catalog = catalog[catalog['costo_unitario'] > 0]
marketing = marketing[marketing['gasto'] >= 0]

In [8]:
# 5. Verificación de consistencia de montos
orders['monto_total'] = (orders['cantidad'] * orders['precio_unitario']) - orders['monto_descuento']
orders = orders[orders['monto_total'] >= 0]

# 6. Revisión y estandarización de variables categóricas
orders['pais'] = orders['pais'].str.strip().str.title()
marketing['pais'] = marketing['pais'].str.strip().str.title()

In [9]:
# Exportación de datasets con la limpieza previa realizada
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

# 💰 Análisis de rentabilidad del negocio 

**Objetivo:** Calcular los indicadores financieros clave (Ingresos totales, Costos operativos, Gasto en Marketing y Ganancia Neta) para evaluar la salud financiera general y la viabilidad del modelo de negocio.

--- **Cálculo de KPIs principales:** ---

In [10]:
# Unión de dataset "Orders" y "Catalog" para traer el costo_unitario de cada producto
orders_merged = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')

print("📊 PARTE 1: RENTABILIDAD DEL NEGOCIO")
print("-" * 40)

# Ingreso total (revenue)
ingreso_total = orders_merged['monto_total'].sum()

# Costo total (cantidad comprada X costo unitario del producto)
costo_total = (orders_merged['cantidad'] * orders_merged['costo_unitario']).sum()

# Inversión en marketing
inversion_marketing = marketing['gasto'].sum()

# Rentabilidad (Profit = Ingresos - Costo - Marketing)
profit = ingreso_total - costo_total - inversion_marketing

print(f"Ingreso Total (Revenue): \033[1m${ingreso_total:,.2f}\033[0m")
print(f"Costo Total: \033[1m${costo_total:,.2f}\033[0m")
print(f"Inversión en Marketing: \033[1m${inversion_marketing:,.2f}\033[0m")
print(f"Ganancia Neta (Profit): \033[1m${profit:,.2f}\033[0m")
print(f"¿Es rentable?: \033[1m{'Si' if profit > 0 else 'No'}\033[0m")

📊 PARTE 1: RENTABILIDAD DEL NEGOCIO
----------------------------------------
Ingreso Total (Revenue): $51,966,982.37
Costo Total: $43,124,069.01
Inversión en Marketing: $2,871,843.53
Ganancia Neta (Profit): $5,971,069.83
¿Es rentable?: Si


In [11]:
print("\n📈 PARTE 2: COMPORTAMIENTO DE VENTAS")
print("-" * 40)
# Agrupamiento por "id_pedido" para consolidar órdenes con múltiples productos
ordenes_agrupadas = orders_merged.groupby('id_pedido').agg(ticket_total=('monto_total', 'sum'), cantidad_total=('cantidad', 'sum'))

# Ticket promedio por orden
ticket_promedio = ordenes_agrupadas['ticket_total'].mean()

# Cantidad promedio de productos por orden
cantidad_promedio = ordenes_agrupadas['cantidad_total'].mean()

# Producto más vendido (volumen de unidades)
producto_mas_vendido = orders_merged.groupby('nombre_producto')['cantidad'].sum().idxmax()
cantidad_producto_top = orders_merged.groupby('nombre_producto')['cantidad'].sum().max()

print(f"Ticket Promedio por Orden: \033[1m${ticket_promedio:,.2f}\033[0m")
print(f"Cantidad Prom. de Productos por Orden: \033[1m{cantidad_promedio:.2f} unidades\033[0m")
print(f"Producto Más Vendido: \033[1m{producto_mas_vendido} ({cantidad_producto_top:,.0f} unidades)\033[0m")

# Gasto de marketing por canal
print("\nGasto de Marketing por Canal:")
gasto_por_canal = marketing.groupby('canal')['gasto'].sum().reset_index()

# Organización de mayor a menor gasto 
gasto_por_canal = gasto_por_canal.sort_values(by='gasto', ascending=False)

for index, row in gasto_por_canal.iterrows():
    print(f"- {row['canal']}: \033[1m${row['gasto']:,.2f}\033[0m")


📈 PARTE 2: COMPORTAMIENTO DE VENTAS
----------------------------------------
Ticket Promedio por Orden: $2,083.18
Cantidad Prom. de Productos por Orden: 7.12 unidades
Producto Más Vendido: Laptop-Gaming-16GB (144,198 unidades)

Gasto de Marketing por Canal:
- social: $918,043.21
- organic: $913,533.01
- paid_search: $863,088.21
- Unknown: $177,179.10


---

# 📉 Análisis de funnel de conversión

**Objetivo:** Identificar los puntos de fricción y descubrir en qué etapa del proceso de compra se pierde la mayor cantidad de usuarios.

In [12]:
db_config = {
    'user': 'TU_USUARIO',
    'pwd': 'TU_PASSWORD',
    'host': 'TU_HOST',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [13]:
# Exploración de tabla "events"
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [14]:
# Totales del funnel
query_totals = '''
SELECT
    nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM
    events
GROUP BY
    nombre_evento
ORDER BY
    usuarios_unicos DESC;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [15]:
# Conversiones
query_conversion = '''
WITH funnel_steps AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
    FROM 
        events
    GROUP BY 
        nombre_evento
)
SELECT
    nombre_evento,
    usuarios_unicos,
    LAG(usuarios_unicos) OVER (ORDER BY usuarios_unicos DESC) AS etapa_anterior,
    ROUND(usuarios_unicos * 100.0 / LAG(usuarios_unicos) OVER (ORDER BY usuarios_unicos DESC), 2) AS tasa_conversion_paso,
    ROUND(usuarios_unicos * 100.0 / FIRST_VALUE(usuarios_unicos) OVER (ORDER BY usuarios_unicos DESC), 2) AS tasa_conversion_total
FROM
    funnel_steps
ORDER BY
    usuarios_unicos DESC;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,etapa_anterior,tasa_conversion_paso,tasa_conversion_total
0,first_visit,7796,NaN,NaN,100.00
1,add_to_cart,7634,7796.0,97.92,97.92
2,select_item,7582,7634.0,99.32,97.26
3,begin_checkout,7208,7582.0,95.07,92.46
4,add_payment_info,6250,7208.0,86.71,80.17
5,purchase,6240,6250.0,99.84,80.04


---

In [16]:
# Exportación de resultados del funnel para Dashboard en Power BI
conversion.to_csv('funnel_conversiones.csv', index=False)

# 🔄 Análisis de Retención de Usuarios (Cohortes)

**Objetivo:** Evaluar la lealtad de los clientes midiendo qué porcentaje de usuarios regresa a la plataforma en los meses posteriores a su primera visita.

In [17]:
# Exploración de la tabla "users"
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [18]:
# Exploración de la  tabla "user_activity"
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [19]:
# Retención por cohortes
query_cohort_retention_final = '''
WITH cohortes AS (
    -- 1. Tamaño real de la cohorte (total de usuarios registrados por mes)
    SELECT 
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_cohorte,
        COUNT(DISTINCT id_usuario) AS clientes_iniciales
    FROM 
        users
    GROUP BY 
        mes_cohorte
),
actividad_semanal AS (
    -- 2. Diferencia en semanas y extracción del indicador de actividad
    SELECT 
        a.id_usuario,
        a.activo,
        DATE_TRUNC('month', CAST(u.fecha_registro AS DATE)) AS mes_cohorte,
        (CAST(a.fecha_actividad AS DATE) - CAST(u.fecha_registro AS DATE)) / 7 AS semanas_desde_registro
    FROM 
        user_activity a
    JOIN 
        users u ON a.id_usuario = u.id_usuario
),
retencion_semanal AS (
    -- 3. Contamos usuarios únicos activos SOLO si la columna activo es 1
    SELECT 
        mes_cohorte,
        COUNT(DISTINCT CASE WHEN semanas_desde_registro = 1 AND activo = 1 THEN id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN semanas_desde_registro = 2 AND activo = 1 THEN id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN semanas_desde_registro = 3 AND activo = 1 THEN id_usuario END) AS retenido_w3
    FROM 
        actividad_semanal
    GROUP BY 
        mes_cohorte
)
-- 4. Cálculo final de porcentajes
SELECT 
    TO_CHAR(c.mes_cohorte, 'YYYY-MM') AS cohorte,
    c.clientes_iniciales,
    ROUND(r.retenido_w1 * 100.0 / NULLIF(c.clientes_iniciales, 0), 2) AS semana_1,
    ROUND(r.retenido_w2 * 100.0 / NULLIF(c.clientes_iniciales, 0), 2) AS semana_2,
    ROUND(r.retenido_w3 * 100.0 / NULLIF(c.clientes_iniciales, 0), 2) AS semana_3
FROM 
    cohortes c
LEFT JOIN 
    retencion_semanal r ON c.mes_cohorte = r.mes_cohorte
ORDER BY 
    c.mes_cohorte;
'''
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,clientes_iniciales,semana_1,semana_2,semana_3
0,2025-01,1627,42.84,41.06,40.32
1,2025-02,1444,42.31,42.17,43.98
2,2025-03,1636,41.38,43.09,42.18
3,2025-04,1606,42.34,43.40,41.28
4,2025-05,1687,41.20,40.07,41.85


---

In [20]:
# Exportación de resultados de retención de cohortes
cohorte_final.to_csv('retencion_cohortes.csv', index=False)

# 🧪 Validación de test estadístico

**Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.


Hipótesis estadística
   - **H₀ (Hipótesis nula):** Las tasas de conversión de ambos grupos son iguales (el cambio en la UI del checkout NO genera impacto).
   - **H₁ (Hipótesis alternativa):** Las tasas de conversión entre los grupos son diferentes (el cambio en la UI del checkout SÍ genera impacto).
   
**Test estadístico:** Z-test para proporciones. 
*Nivel de significancia alpha:* 0.05 (5% de margen de error estándar).  

In [21]:
# Uploading and checking dataset
experiment = pd.read_csv('/datasets/experiment_checkout_ui.csv')
print("\033[1mVista previa de los datos:\033[0m")
display(experiment.head())

# Agrupación de datos para sacar totales y conversiones por grupo
columna_grupo = 'variante'

resumen_ab = experiment.groupby(columna_grupo)['convirtio'].agg(['count', 'sum'])
resumen_ab.columns = ['usuarios_totales', 'compras']
resumen_ab['tasa_conversion'] = resumen_ab['compras'] / resumen_ab['usuarios_totales']

print("\n\033[1mResumen del Test A/B:\033[0m")
display(resumen_ab)

Vista previa de los datos:


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12



Resumen del Test A/B:


,usuarios_totales,compras,tasa_conversion
variante,,,
control,4965,779,0.156898
tratamiento,5035,820,0.162860


In [22]:
# Aplicación del test estadístico (Z-test para proporciones)
compras_array = resumen_ab['compras'].values
totales_array = resumen_ab['usuarios_totales'].values

z_stat, p_value = proportions_ztest(compras_array, totales_array)

# Interpretación del resultado
alpha = 0.05
print("\n\033[1m--- Conclusión del Test Estadístico ---\033[0m")
print(f"Valor p (p-value): {p_value:.4f}")
print(f"Nivel de significancia (alpha): {alpha}")

if p_value < alpha:
    print("Resultado: \033[1mRechazamos la hipótesis nula (H0).\033[0m")
    print("Interpretación: Hay una diferencia estadísticamente significativa. El cambio en la UI SÍ generó un impacto real en la conversión.")
else:
    print("Resultado: \033[1mNo podemos rechazar la hipótesis nula (H0).\033[0m")
    print("Interpretación: NO hay evidencia suficiente para decir que la nueva UI impactó la conversión (los resultados podrían ser obra del azar).")


--- Conclusión del Test Estadístico ---
Valor p (p-value): 0.4161
Nivel de significancia (alpha): 0.05
Resultado: No podemos rechazar la hipótesis nula (H0).
Interpretación: NO hay evidencia suficiente para decir que la nueva UI impactó la conversión (los resultados podrían ser obra del azar).


In [23]:
# Exportación para Power BI
resumen_ab.reset_index().to_csv('resultados_test_ab.csv', index=False)

---

# 📊Visualización de resultados (Dashboard en BI)

**Objetivo**: Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

 📎**Enlace a la caperta en Google Drive con el archivo del Dashboard en Power BI y el archivo de Excel con los _clean.csv**
 
 *https://drive.google.com/drive/folders/1XSxFPvXT7BKF9Bc1gUAQsZSAoYLx0dcr?usp=sharing*